In [2]:
import aiohttp
import asyncio
import nest_asyncio
import pandas as pd
from datetime import datetime, timedelta
from tqdm.asyncio import tqdm_asyncio
import os
from dotenv import load_dotenv

# --- Загружаем API-ключ ---
load_dotenv("configs.env")

nest_asyncio.apply()
API_KEY = os.getenv("API_KEY") 
OUTPUT_FILE = "data/tmdb_movies_2020_04.csv"

BASE_URL = "https://api.themoviedb.org/3"
DISCOVER_URL = f"{BASE_URL}/discover/movie"
DETAIL_URL = f"{BASE_URL}/movie/{{}}"
CREDITS_URL = f"{BASE_URL}/movie/{{}}/credits"
KEYWORDS_URL = f"{BASE_URL}/movie/{{}}/keywords"

# --- ограничим параллельность ---
semaphore = asyncio.Semaphore(15)

# --- создаём папку для сохранения ---
os.makedirs("data", exist_ok=True)


async def safe_request(session, url, params):
    """Безопасный запрос с повторными попытками и обработкой лимитов."""
    async with semaphore:
        for attempt in range(3):
            try:
                async with session.get(url, params=params, timeout=20) as response:
                    if response.status == 200:
                        return await response.json()
                    elif response.status == 429:
                        print("⏳ Превышен лимит запросов. Ожидание 10 секунд...")
                        await asyncio.sleep(10)
                    else:
                        print(f"⚠️ Ошибка {response.status} при запросе {url}")
                        await asyncio.sleep(2)
            except Exception as e:
                print(f"❌ Ошибка при запросе {url}: {e}")
                await asyncio.sleep(2)
    return {}


async def fetch_movies_page(session, page, start_date, end_date):
    """Загрузка одной страницы фильмов."""
    params = {
        "api_key": API_KEY,
        "language": "en-US",
        "sort_by": "primary_release_date.asc",
        "include_adult": "false",
        "include_video": "false",
        "release_date.gte": start_date,
        "release_date.lte": end_date,
        "page": str(page)
    }
    data = await safe_request(session, DISCOVER_URL, params)
    return data.get("results", [])


async def fetch_all_movie_ids(session, start_date, end_date):
    """Загрузка всех ID фильмов за месяц."""
    if not start_date or not end_date:
        print(f"⚠️ Пропущены даты: start_date={start_date}, end_date={end_date}")
        return []
    params = {
        "api_key": API_KEY,
        "language": "en-US",
        "sort_by": "primary_release_date.asc",
        "include_adult": "false",
        "include_video": "false",
        "release_date.gte": start_date,
        "release_date.lte": end_date,
        "page": "1"
    }

    first_page = await safe_request(session, DISCOVER_URL, params)
    total_pages = min(first_page.get("total_pages", 1), 500)
    movies = first_page.get("results", [])

    tasks = [fetch_movies_page(session, page, start_date, end_date) for page in range(2, total_pages + 1)]
    for task in tqdm_asyncio.as_completed(tasks, total=len(tasks), desc=f"📥 Pages {start_date[:7]}"):
        movies.extend(await task)

    return [m["id"] for m in movies if "id" in m]


async def fetch_movie(session, movie_id):
    """Загрузка деталей одного фильма."""
    params = {"api_key": API_KEY, "language": "en-US"}
    urls = {
        "basic": DETAIL_URL.format(movie_id),
        "credits": CREDITS_URL.format(movie_id),
        "keywords": KEYWORDS_URL.format(movie_id)
    }

    basic_task = safe_request(session, urls["basic"], params)
    credits_task = safe_request(session, urls["credits"], params)
    keywords_task = safe_request(session, urls["keywords"], params)

    basic, credits, keywords = await asyncio.gather(basic_task, credits_task, keywords_task)

    return {
        "id": movie_id,
        "belongs_to_collection": basic.get("belongs_to_collection"),
        "budget": basic.get("budget"),
        "genres": basic.get("genres"),
        "homepage": basic.get("homepage"),
        "imdb_id" : basic.get("imdb_id"),
        "original_language": basic.get("original_language"),         
        "original_title": basic.get("original_title"),       
        "overview": basic.get("overview"),
        "popularity": basic.get("popularity"),
        "poster_path": basic.get("poster_path"),
        "production_companies": basic.get("production_companies"),
        "production_countries": basic.get("production_countries"),
        "release_date": basic.get("release_date"), 
        "runtime": basic.get("runtime"),
        "spoken_languages": basic.get("spoken_languages"),
        "status": basic.get("status"),
        "tagline": basic.get("tagline"),
        "title": basic.get("title"),        
        "keywords": keywords.get("keywords"),
        "cast": credits.get("cast"),
        "crew": credits.get("crew"),                       
        "revenue": basic.get("revenue"),                             
        "primary_release_date": basic.get("release_date"),
        "vote_average": basic.get("vote_average"),
        "vote_count": basic.get("vote_count")
    }


async def process_month(session, start_date, end_date):
    """Сбор и сохранение данных за один месяц."""
    movie_ids = await fetch_all_movie_ids(session, start_date, end_date)
    print(f"🎬 Найдено {len(movie_ids)} фильмов за {start_date[:7]}")

    if not movie_ids:
        return pd.DataFrame()

    tasks = [fetch_movie(session, movie_id) for movie_id in movie_ids]
    results = []
    for coro in tqdm_asyncio.as_completed(tasks, total=len(tasks), desc=f"🎞 Fetch {start_date[:7]}"):
        results.append(await coro)

    df = pd.DataFrame(results)
    df = df.drop_duplicates("id")
    df = df[df["revenue"] != 0]  # удаляем фильмы без выручки

    # Сохранение помесячно (append)
    df.to_csv(OUTPUT_FILE, mode="a", index=False, encoding="utf-8-sig", header=not pd.io.common.file_exists(OUTPUT_FILE))
    print(f"💾 Сохранено {len(df)} фильмов за {start_date[:7]}")
    return df


async def main():
    async with aiohttp.ClientSession() as session:
        start_date = "2020-04-01"
        end_date = "2020-04-30"
        print(f"\n🚀 Обработка месяца: {start_date[:7]} ({start_date} → {end_date})")
        await process_month(session, start_date, end_date)
        await asyncio.sleep(2)


# --- Запуск ---
asyncio.run(main())

print("\n✅ Сбор данных завершён! Все фильмы сохранены в", OUTPUT_FILE)


🚀 Обработка месяца: 2020-04 (2020-04-01 → 2020-04-30)


📥 Pages 2020-04: 100%|██████████████████████████████████████████████████████████████| 152/152 [00:02<00:00, 59.45it/s]


🎬 Найдено 3053 фильмов за 2020-04


🎞 Fetch 2020-04: 100%|█████████████████████████████████████████████████████████████| 3053/3053 [03:28<00:00, 14.63it/s]


💾 Сохранено 279 фильмов за 2020-04

✅ Сбор данных завершён! Все фильмы сохранены в data/tmdb_movies_2020_04.csv
